### Connection Mongo DB

In [ ]:
pip install pymongo

In [ ]:
import pymongo

In [ ]:
from pymongo import MongoClient

conn = MongoClient("mongodb://localhost:27017")

db = conn["university"]

In [ ]:
print(db.list_collection_names())

### Extract Data

In [ ]:
import pandas as pd

In [ ]:
dfstudents = pd.DataFrame(list(db.students.find({})))
dfstudents

In [ ]:
dfattendances = pd.DataFrame(list(db.attendances.find({})))
dfattendances

In [ ]:
dfgrades = pd.DataFrame(list(db.grades.find({})))
dfgrades

### Transform Data

In [ ]:
dfstudents.info()

In [ ]:
dfstudents.isnull().sum()

In [ ]:
dfstudents['email'] = dfstudents['email'].fillna('UNKNOWN')
dfstudents['phone'] = dfstudents['phone'].fillna('UNKNOWN')
dfstudents['gender'] = dfstudents['gender'].fillna('UNKNOWN')
dfstudents['level'] = dfstudents['level'].fillna(dfstudents['level'].mode()[0])
dfstudents['age'] = dfstudents['age'].fillna(dfstudents['age'].mode()[0])


In [ ]:
dfstudents['age'] = dfstudents['age'].astype('Int64')

In [ ]:
dfstudents['level'] = dfstudents['level'].astype('Int64')

In [ ]:
dfstudents['city'].value_counts()

In [ ]:
dfstudents.isnull().sum()

In [ ]:
dfattendances.info()

In [ ]:
dfattendances['date'] = pd.to_datetime(dfattendances['date'], format='%Y-%m-%d')

In [ ]:
dfattendances.isnull().sum()

In [ ]:
dfattendances['status'].value_counts()

In [ ]:
dfattendances['status'] = dfattendances['status'].replace({'present':'Present'}, inplace=True)

In [ ]:
dfattendances['status'].value_counts()

In [ ]:
dfattendances['status'] = dfattendances['status'].fillna(dfattendances['status'].mode()[0])

In [ ]:
dfattendances['note'] = dfattendances['note'].fillna('No note')

In [ ]:
dfattendances['note'].value_counts()

In [ ]:
dfattendances.isnull().sum()

In [ ]:
dfattendances.info()

In [ ]:
dfgrades.info()

In [ ]:
dfgrades['math'] = dfgrades['math'].astype('Int64')

In [ ]:
dfgrades.isnull().sum()

In [ ]:
dfgrades['note'] = dfgrades['note'].fillna('No note')

In [ ]:
dfgrades.isnull().sum()

### Load Data

In [ ]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://DESKTOP-JKQGEOC\\SQLSERVER2025/universityDWH"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

In [ ]:

math_df =dfgrades[['student_id', 'math']].copy()
math_df = math_df.rename(columns={'math': 'grade'})
math_df['subject'] = 'math'

python_df = dfgrades[['student_id', 'python']].copy()
python_df = python_df.rename(columns={'python': 'grade'})
python_df['subject'] = 'python'

grades = pd.concat(
    [math_df, python_df],
    ignore_index=True
)

print(grades.head())

In [ ]:
StudentPerformance = pd.merge(dfstudents, grades, on='student_id', how='left')
StudentPerformance = pd.merge(StudentPerformance, dfattendances, on='student_id', how='left')

In [ ]:
StudentPerformance.columns

In [ ]:
StudentPerformance = StudentPerformance[['student_id', 'name','age', 'city', 'subject' ,'grade', 'date', 'status']]

In [ ]:
StudentPerformance = StudentPerformance.rename(columns={'name': 'student_name', 'date':'attendance_date', 'status':'attendance_status'})

In [ ]:
StudentPerformance.columns

In [ ]:
StudentPerformance.head(5)

In [ ]:
StudentPerformance.to_sql('StudentPerformance', con=engine, if_exists='append', index=False)